# MedGemma 1.5 4B — Colab GPU Runner (Hybride)

Lanceur minimal : le code source reste versionné dans le repo Git ; Colab sert uniquement de banc GPU (**L4** pour la baseline, **A100 40 Go** pour QLoRA). Pensez à **déconnecter le runtime** dès la fin du run.

In [ ]:
# 1. Vérification GPU (bfloat16 matériel SM >= 8.0 requis) & montage Google Drive
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'Aucun GPU détecté : choisir L4 ou A100.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {gpu_name} ({vram_gb:.1f} Go VRAM) | Compute Capability: {major}.{minor}')
assert major >= 8, (
    f'GPU {gpu_name} (SM {major}.{minor}) sans bfloat16 matériel natif : '
    'passez sur L4 ou A100 dans Exécution > Modifier le type d\'exécution.'
)

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/MedGemma_Dermato'

In [ ]:
# 2. Authentification Hugging Face via Colab Secrets (jamais de token en dur)
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Ajouter HF_TOKEN dans les Colab Secrets (icône Clé à gauche).'
login(token=hf_token, add_to_git_credential=False)
print('Authentification Hugging Face OK.')

In [ ]:
# 3. Clone/mise à jour du repo GitHub et installation en mode éditable avec dépendances GPU
import os
import subprocess

REPO_URL = 'https://github.com/sailwaelan-dotcom/medgemma-dermato.git'
REPO_DIR = '/content/medgemma-dermato'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)
subprocess.run(
    ['python', '-c', 'import medgemma_dermato.data.isic_ingest; print("Package medgemma_dermato OK.")'],
    check=True,
)

In [ ]:
# 4. Smoke test cross-hospitalier (réutilise data/isic_smoke sur Drive si présent) + baseline zéro-shot
import os
import subprocess

SMOKE_DIR = '/content/drive/MyDrive/MedGemma_Dermato/data/isic_smoke'
if not os.path.exists(f'{SMOKE_DIR}/test.jsonl'):
    subprocess.run(
        [
            'python', 'scripts/build_dataset.py',
            '--mode', 'cross-hospital',
            '--commercial-only',
            '--max-items', '150',
            '--skip-exclude-fetch',
            '--output-dir', SMOKE_DIR,
        ],
        check=True,
    )

subprocess.run(
    [
        'python', 'scripts/run_baseline.py',
        '--manifest', f'{SMOKE_DIR}/test.jsonl',
        '--image-dir', '/content/drive/MyDrive/MedGemma_Dermato/data/images',
        '--output-dir', '/content/drive/MyDrive/MedGemma_Dermato/outputs/baseline_smoke',
        '--limit', '20',
    ],
    check=True,
)

In [ ]:
# 5. Entraînement QLoRA (A100 40 Go) avec checkpoints persistés sur Drive
# NE PAS LANCER avant d'avoir analysé eval_report.md de la cellule 4 et préparé les cibles QLoRA sur A100.
import subprocess

subprocess.run(
    [
        'python', 'scripts/train_qlora.py',
        '--train-manifest', '/content/drive/MyDrive/MedGemma_Dermato/data/isic_smoke/train.jsonl',
        '--val-manifest', '/content/drive/MyDrive/MedGemma_Dermato/data/isic_smoke/val.jsonl',
        '--image-dir', '/content/drive/MyDrive/MedGemma_Dermato/data/images',
        '--output-dir', '/content/drive/MyDrive/MedGemma_Dermato/checkpoints/qlora_run1',
    ],
    check=True,
)

# IMPORTANT : déconnecter le runtime immédiatement après la fin du run :
# from google.colab import runtime; runtime.unassign()